<a id="part-i"></a>
# Part I — Overview and navigation

This runnable lesson uses **invented teaching data**, not real observations or Oracle research
results. It demonstrates the complete repository workflow before you adapt the general
authoring template to your own project.

| Part | Reader task |
|---|---|
| [I](#part-i) | Understand the example and execution order |
| [II](#part-ii) | Identify sources, metadata and dictionary |
| [III](#part-iii) | Retrieve the exact code and acquire pinned data |
| [IV](#part-iv) | Inspect queried data |
| [V](#part-v) | Process with documented rules |
| [VI](#part-vi) | Inspect processed records and their meanings |
| [VII](#part-vii) | Reproduce a descriptive summary |
| [VIII](#part-viii) | Inspect validation and compare reference outputs |

Open in Google Colab, save a copy, then run code cells in order using the play button or
Shift+Enter. Restart the runtime before repeating with a different code version.
A red error means stop and read its message; later outputs may be stale.
No credentials, external datasets or third-party installs are needed. Runtime files are
temporary; download chosen outputs before disconnecting.

One record represents an invented temperature observation with an invented group and time.
Celsius and kelvin are two temperature scales; this example adds 273.15 to convert between
them. A checksum tests file identity, not scientific correctness. A commit identifies a code version.

**First author:** replace the example with justified project methods and document scope,
permissions, expected outputs and an independent reproduction in the general authoring notebook.


<a id="part-ii"></a>
## Part II — Data sources, metadata, dictionary and URLs

| Item | Teaching resource |
|---|---|
| Source | [Eight invented observations](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/data/data_source/demo/observations.csv) |
| Data version | teaching-example-1 |
| Origin | Synthetic teaching values; no field collection or human subjects |
| Coverage | Eight rows including one exact duplicate and one deliberately invalid value |
| Dictionary | [Field definitions](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/metadata/demo_dictionary.csv) |
| Rights | [Existing MIT licence](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/LICENSE) for this teaching material |
| Source hash | Verified by configs/demo.json |

Fields are observation_id, group_id, timestamp_utc, temperature_c and origin.
An empty temperature means missing; zero would be a different value.
The output adds temperature_k and value_status.

**First author:** supply the actual population, sources, versions, rights, field meanings,
units, missing-value semantics, sensitive information and collection/annotation protocol.
See [Scientific Data Methods/Data Records](https://www.nature.com/sdata/submission-guidelines)
and [NeurIPS dataset metadata](https://neurips.cc/Conferences/2026/EvaluationsDatasetsHosting).


<a id="part-iii"></a>
## Part III — Query data

The next cell retrieves a specified code commit and creates a new run directory.
The acquisition command then copies the local teaching fixture and verifies its configured
SHA-256. The same query module supports explicitly configured public HTTPS archives.
It does not invent live API authentication, pagination or collection logic.

Code: [query_data](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/code/query_data/query.py).
**Expected output:** queried_data/observations.csv and reports/acquisition.json.
**First author:** document actual queries, versions, time zones, pagination, failures,
rights and scope before substituting another data source.


In [ ]:
from pathlib import Path
import csv, hashlib, json, re, subprocess, sys, uuid

CODE_COMMIT = "27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2"
if not re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    raise ValueError("Supply the validated repository commit before running this notebook.")
BASE = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = BASE / ("Oracle4SD-demo-" + uuid.uuid4().hex[:8])
subprocess.run(["git", "clone", "--no-checkout",
                "https://github.com/sunshineluyao/Oracle4SD.git", str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", CODE_COMMIT], check=True)
actual = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
if actual != CODE_COMMIT:
    raise ValueError("Code revision mismatch")
RUN_DIR = REPO_DIR / "runs" / ("colab-" + uuid.uuid4().hex[:8])
RUN_DIR.mkdir(parents=True)
CONFIG = REPO_DIR / "configs/demo.json"

def stage(folder, filename):
    subprocess.run([sys.executable, str(REPO_DIR / "code" / folder / filename),
                    "--config", str(CONFIG), "--run-dir", str(RUN_DIR)], check=True)

def show_csv(path, limit=10):
    with Path(path).open(newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    try:
        from IPython.display import HTML, display
        from html import escape
        fields = list(rows[0]) if rows else []
        header = "".join("<th>" + escape(s) + "</th>" for s in fields)
        body = "".join("<tr>" + "".join("<td>" + escape(row[s]) + "</td>" for s in fields) + "</tr>"
                       for row in rows[:limit])
        display(HTML("<table><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"))
    except ImportError:
        print(json.dumps(rows[:limit], indent=2))
    print("Total records:", len(rows))
    return rows

print("Pinned code:", actual)
print("Runtime Python:", sys.version.split()[0])
print("Run folder:", RUN_DIR)


In [ ]:
stage("query_data", "query.py")
print((RUN_DIR / "reports/acquisition.json").read_text())


<a id="part-iv"></a>
## Part IV — Queried data

Inspect the eight acquired rows. Find the exact duplicate, missing value and deliberately
out-of-range value. These are intentional teaching cases.
Code and protocol: [acquisition documentation](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/data/queried_data/README.md).
**First author:** explain each raw field, source trace, anomaly and sample/full scope.
An inspection preview alone does not demonstrate completeness.


In [ ]:
raw_rows = show_csv(RUN_DIR / "queried_data/observations.csv")
raw_hash_before = hashlib.sha256((RUN_DIR / "queried_data/observations.csv").read_bytes()).hexdigest()


<a id="part-v"></a>
## Part V — Process data

The teaching adapter removes exact duplicates, excludes values outside an arbitrary
−50 to 60 Celsius range, retains missing values, and converts finite values to kelvin.
The range is a teaching choice, not a scientific recommendation.
Code: [process.py](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/code/process_data/process.py).
**Expected:** six rows and a decision log. Raw data remain unchanged.
**First author:** replace these rules with justified domain methods, document information
loss and reconcile all count changes, including joins and aggregation.


In [ ]:
stage("process_data", "process.py")
raw_hash_after = hashlib.sha256((RUN_DIR / "queried_data/observations.csv").read_bytes()).hexdigest()
if raw_hash_after != raw_hash_before:
    raise ValueError("Raw input changed")
print((RUN_DIR / "reports/processing.json").read_text())


<a id="part-vi"></a>
## Part VI — Processed data

Inspect the six retained records and full dictionary. One temperature remains missing.
For example, demo-001 changes from 10 Celsius to 283.15 kelvin.
Documentation: [processed-data README](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/data/processed_data/README.md).
**First author:** provide the final manifest, dictionary, relationships, versions, data licence
and persistent identifiers. For a NeurIPS dataset contribution, complete and validate
actual Croissant/RAI metadata; the repository's template JSON is not submission metadata.


In [ ]:
processed_rows = show_csv(RUN_DIR / "processed_data/observations.csv")
dictionary = show_csv(REPO_DIR / "metadata/demo_dictionary.csv")


<a id="part-vii"></a>
## Part VII — Analyze data

Compute a descriptive summary of the teaching values.
Code: [analyze.py](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/code/analyze_data/analyze.py).
**Expected:** six records, five numeric values, one missing value; mean 14.0000 Celsius.
The denominator for this mean is five, not six.
**First author:** define the question, denominator, uncertainty and interpretation of every
output. Keep the Data Descriptor within its data-description and technical-quality scope.


In [ ]:
stage("analyze_data", "analyze.py")
summary = show_csv(RUN_DIR / "analysis/summary.csv")


<a id="part-viii"></a>
## Part VIII — Technical validation

Inspect software checks and compare deterministic outputs with the governed teaching
reference. This comparison does not establish measurement accuracy or a real scientific
population; those dimensions remain not evaluated.
Code: [validate.py](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/code/technical_validation/validate.py).
Requirements: [Scientific Data Technical Validation](https://www.nature.com/sdata/submission-guidelines);
[NeurIPS review](https://neurips.cc/Conferences/2026/EvaluationsDatasetsReviewerGuidelines).

**First author:** supply actual references, audit populations, sampling, metrics, criteria,
uncertainty, failure cases and independent-run evidence. Record validators and loading
scope separately from scientific validity. Do not mark unchecked dimensions passed.


In [ ]:
stage("technical_validation", "validate.py")
report = json.loads((RUN_DIR / "reports/validation.json").read_text())
print(json.dumps(report, indent=2))
reference = json.loads((REPO_DIR / "tests/reference/demo_manifest.json").read_text())
if reference.get("data_version") != json.loads(CONFIG.read_text())["data_version"]:
    raise ValueError("Reference data version differs from the configured release.")
comparison = []
for item in reference["files"]:
    path = RUN_DIR / item["path"]
    matches = path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == item["sha256"]
    comparison.append({"file": item["path"], "matches": matches})
print(json.dumps(comparison, indent=2))
if not comparison or not all(item["matches"] for item in comparison):
    raise ValueError("Reference mismatch: investigate rather than overwriting expected outputs.")
print("Teaching output comparison passed. Scientific validation remains NOT EVALUATED.")


## Finish and adapt

For a real project, fill the [general authoring notebook](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/notebooks/Scientific_Data_Colab_Tutorial_Template.ipynb)
and [first-author guide](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/docs/FIRST_AUTHOR_GUIDE.md).
Ask an unfamiliar reader to repeat the documented workflow and record questions.
Hugging Face packaging is a separate [local preparation step](https://github.com/sunshineluyao/Oracle4SD/blob/27a7e1e2a1ff9ca1afcf42f13476cc5cfbadf0e2/docs/HUGGING_FACE.md);
no upload occurs in this notebook.

Download any teaching outputs you want to retain through Colab's Files pane.
See [Colab's FAQ](https://research.google.com/colaboratory/faq.html) for runtime and file behavior.
[Back to navigation](#part-i).
